# JavaScript concepts behind the widget host `notebook.js_concepts`

This notebook runs on the Deno kernel. Each cell shows one concept from the widget isolation code. Run the cells in order.

## Emitter and handlers `notebook.js_concepts.emitter`

The emitter keeps a table of handlers. `emit` calls each handler of a name, in the order of registration. A handler is only a function.

In [1]:
class Emitter {
  #table = new Map<string, Array<(...args: unknown[]) => void>>()
  on(name: string, fn: (...args: unknown[]) => void) {
    const list = this.#table.get(name) ?? []
    list.push(fn)
    this.#table.set(name, list)
  }
  off(name: string, fn: (...args: unknown[]) => void) {
    this.#table.set(name, (this.#table.get(name) ?? []).filter((h) => h !== fn))
  }
  emit(name: string, ...args: unknown[]) {
    for (const fn of this.#table.get(name) ?? []) fn(...args)
  }
}

const e = new Emitter()
const first = (x: unknown) => console.log('first handler got', x)
e.on('tick', first)
e.on('tick', (x) => console.log('second handler got', x))
e.emit('tick', 1)
e.off('tick', first)
e.emit('tick', 2)

first handler got 1
second handler got 1
second handler got 2


## Handler leak on reload `notebook.js_concepts.leak`

A global registration has no owner. Each reload adds one more handler, as the HMR safety spike measured for `process.on`.

In [2]:
const globalBus = new Emitter()
let count = 0
function loadPluginWithRawHandler() {
  globalBus.on('event', () => { count++ })   // no undo step
}
for (let reload = 0; reload < 4; reload++) loadPluginWithRawHandler()
count = 0
globalBus.emit('event')
console.log('handlers that ran for one event after 4 reloads:', count)

handlers that ran for one event after 4 reloads: 4


## Effect with a disposer `notebook.js_concepts.effect`

A Cordis effect returns a disposer. The owner runs the disposers in reverse order (LIFO) when it unloads. Thus a reload adds nothing.

In [3]:
class Fiber {
  #disposers: Array<() => void> = []
  effect(fn: () => () => void) { this.#disposers.push(fn()) }
  dispose() { while (this.#disposers.length) this.#disposers.pop()!() }
}

const bus2 = new Emitter()
let hits = 0
function loadPluginWithEffect(): Fiber {
  const fiber = new Fiber()
  fiber.effect(() => {
    const h = () => { hits++ }
    bus2.on('event', h)
    return () => bus2.off('event', h)
  })
  return fiber
}
let fiber = loadPluginWithEffect()
for (let reload = 0; reload < 4; reload++) { fiber.dispose(); fiber = loadPluginWithEffect() }
bus2.emit('event')
console.log('handlers that ran for one event after 4 reloads:', hits)

handlers that ran for one event after 4 reloads: 1


## Promise and `.then` `notebook.js_concepts.promise`

A promise is a value that settles once. `.then` is a continuation on that one promise. It runs in a later microtask, after the current call stack is empty.

In [4]:
console.log('1 sync start')
const p = Promise.resolve(42)
p.then((v) => console.log('3 then got', v))
queueMicrotask(() => console.log('4 microtask'))
setTimeout(() => console.log('5 timer, a later turn of the event loop'), 0)
console.log('2 sync end')
await new Promise((r) => setTimeout(r, 10))

1 sync start
2 sync end
3 then got 42
4 microtask
5 timer, a later turn of the event loop


## Last-resort handler for rejections `notebook.js_concepts.unhandled`

When a promise rejects and no code handles it, the runtime emits a global event. Node calls this event `unhandledRejection` on `process`. Deno calls it `unhandledrejection` on `globalThis`. The cell stops the default action, so the kernel stays up.

In [5]:
const onUnhandled = (ev: PromiseRejectionEvent) => {
  ev.preventDefault()
  console.log('global handler saw:', String(ev.reason))
}
globalThis.addEventListener('unhandledrejection', onUnhandled)
Promise.reject(new Error('nobody caught me'))
await new Promise((r) => setTimeout(r, 20))
globalThis.removeEventListener('unhandledrejection', onUnhandled)

global handler saw: Error: nobody caught me


## AsyncLocalStorage and the async chain `notebook.js_concepts.als`

`run(value, fn)` starts a chain. Each timer, promise, and microtask that the chain creates keeps the value. Code outside any `run` sees `undefined`. The blame store in `deepwidgets/widgets/src/isolation.ts` works this way.

In [6]:
import { AsyncLocalStorage } from 'node:async_hooks'
const blame = new AsyncLocalStorage<string>()
const sleep = (ms: number) => new Promise((r) => setTimeout(r, ms))

blame.run('widget-a', async () => {
  await sleep(10)
  setTimeout(() => console.log('timer started by widget-a sees:', blame.getStore()), 5)
})
blame.run('widget-b', () => {
  Promise.resolve().then(() => console.log('then in widget-b sees:', blame.getStore()))
})
console.log('code outside any run sees:', blame.getStore())
await sleep(40)

code outside any run sees: undefined
then in widget-b sees: widget-b
timer started by widget-a sees: widget-a


## Blame from a global handler `notebook.js_concepts.blame`

The global handler gets only the error. It reads the store to learn which widget started the chain that threw.

In Deno, the `unhandledrejection` listener runs outside the chain that rejected. Thus the store is empty, and both failures show "no widget". The deepWidgets host runs on Node. There, the check `tools/widgets-blame.mjs` shows that a throw in a widget timer gets the right widget. This notebook does not test a Node rejection.

In [7]:
const seen: string[] = []
const blameHandler = (ev: PromiseRejectionEvent) => {
  ev.preventDefault()
  seen.push(`${String((ev.reason as Error).message)} -> blamed on ${blame.getStore() ?? 'no widget'}`)
}
globalThis.addEventListener('unhandledrejection', blameHandler)
blame.run('widget-c', () => { setTimeout(() => { Promise.reject(new Error('late failure')) }, 5) })
Promise.reject(new Error('top-level failure'))
await sleep(40)
globalThis.removeEventListener('unhandledrejection', blameHandler)
console.log(seen.join('\n'))

top-level failure -> blamed on no widget
late failure -> blamed on no widget


## Keep the context at the failure point `notebook.js_concepts.snapshot`

A handler that the chain itself attaches runs inside the chain. Thus it reads the right store. `AsyncLocalStorage.snapshot()` captures the context, so code can run in it later.

In [8]:
const report: string[] = []
blame.run('widget-d', () => {
  const inChain = AsyncLocalStorage.snapshot()
  setTimeout(() => {
    Promise.reject(new Error('late failure'))
      .catch((err) => report.push(`${err.message} -> caught in chain, blamed on ${blame.getStore()}`))
  }, 5)
  setTimeout(() => inChain(() => report.push(`snapshot run later sees ${blame.getStore()}`)), 10)
})
await sleep(40)
console.log(report.join('\n'))

late failure -> caught in chain, blamed on widget-d
snapshot run later sees widget-d
